In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path # Working with paths in python

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Import and Analyse Data

First we import the competition dataset into our Kaggle notebook to be used.

In [ ]:
path = Path('/kaggle/input/nlp-getting-started')
# ! pip install -q datasets

Look at the data to see what we files we are given.

In [ ]:
!ls {path}

Import data into dataframe to be used and inspected.

In [ ]:
trn_df = pd.read_csv(path/'train.csv')
tst_df = pd.read_csv(path/'test.csv')

trn_df
trn_df.describe(include='object') # Show some useful statistics

Now that we have the data in a dataframe, we want to create an angregate column called input that we can use for our NLP (Natural Language Processing) model. We can represent the input to the model as something like "TARG1: abatement of pollution; CONT1: A47; ANC1: abatement"

In [ ]:
trn = trn_df.drop(['id', 'target'], axis=1)
tst = tst_df.drop(['id'], axis=1)
trn, tst

In [ ]:
##df['input'] = 'TARG1: ' + df.target + '; CONT1: ' + df.context + '; ANC1: ' + df.anchor
##df.input.head()

# Convert dataset to dictionary
trn_list = trn.to_dict(orient='records')
tst_list = tst.to_dict(orient='records')
trn_list[:2]

In [ ]:
# Convert the dictionary row to a string and use this as our TARG1: ...
trn_txt = [str(row) for row in trn_list]
tst_txt = [str(row) for row in tst_list]
trn_txt[:2], tst_txt[:2]

In [ ]:
# Save the string into the input column
trn_df['input'] = trn_txt
tst_df['input'] = tst_txt
trn_df.input.head(), tst_df.input.head()

## Tokenization

Hugging face Transformers use a `Dataset` object for storing data. Therefore if we want to use Transformers we should create one.

In [ ]:
from datasets import Dataset,DatasetDict

trn_ds = Dataset.from_pandas(trn_df)
tst_ds = Dataset.from_pandas(tst_df)
trn_ds, tst_ds # This is what the datasets look like

We can't pass text directly into our model. A deep learning model expects number inputs, therefore we should do two things.

- Tokenization: Split text into tokens
- Numericalization: Convert each token into a number

The first step is to choose our model, as this changes how we complete these two processes.

In [ ]:
model_nm = 'google-bert/bert-base-uncased' # Trying the google-bert model
# model_nm = 'microsoft/deberta-v3-small'

Now we use `AutoTokenizer` to create a tokenizer appropriate for our model.

In [ ]:
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)

Setup a simple function now to tokenize our inputs

In [ ]:
def tok_func(x): return tokz(x["input"])    

Now run this on every row in our dataset using `map` so it can run in parallel batches.

In [ ]:
trn_tok_ds = trn_ds.map(tok_func, batched=True)
tst_tok_ds = tst_ds.map(tok_func, batched=True)

This will add a new line to the dataset called `input_ids`, holding our tokenized and numericalized data. Let's display our first row of data to have a look.

In [ ]:
row = trn_tok_ds[0]
row['input'], row['input_ids']

Finally, Transformers always assume that our labels have the column name `labels`. In the dataset we are given for the competition it is currently named score, so let's change that.

In [ ]:
trn_tok_ds = trn_tok_ds.rename_columns({'target':'labels'})

Now we're ready to create our validation set

## Validation set

Our data directory contained another files called `test.csv`

In [ ]:
#eval_df = pd.read_csv(path/'test.csv')
#eval_df.describe()

To get the validation set we must split up `train.csv`

In [ ]:
dds = trn_tok_ds.train_test_split(0.25, seed=42)
dds

In [ ]:
#eval_df['input'] = 'TARG1: ' + eval_df.target + '; CONT1: ' + eval_df.context + '; ANC1: ' + eval_df.anchor
#eval_ds = Dataset.from_pandas(eval_df).map(tok_func, batched=True)

## Metrics and correlation

In this competition the metric being used to decide the effectiveness of the model is *F1*.

`Transformers` expect metrics to be returned as a `dict`, since that way the trainer knows what labels to use for what metrics, so let's define a function to do that

In [ ]:
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
from sklearn.metrics import f1_score
def my_metric(eval_pred): 
    preds = np.argmax(eval_pred.predictions, axis=1)  # get predicted class indices
    labels = eval_pred.label_ids
    return {'F1': f1_score(labels, preds, zero_division=0)}

## Training

It's training time!! To train a model in Transformers we need this:

In [ ]:
from transformers import TrainingArguments,Trainer

Now we pick a batch size suitable for our GPU, and a small number of epochs so we can run experiments quickly.

In [ ]:
bs = 8
# epochs = 4
epochs = 1  # for our prac 1 epoch is enough

The most important hyperparameter is the learning rate. Transformers doesn't provide a way to find this learning rate, so we'll just have to use trial and error. The idea is to find the largest value we can, but which doesn't result in training failing.

In [ ]:
lr = 8e-5

Transformers uses the `TrainingArguments` class to set up arguments. Most of this is just boilerplate.

In [ ]:
args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, lr_scheduler_type='cosine', fp16=True,
    eval_strategy="epoch", per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, weight_decay=0.01, report_to='none')

We can now create our model, and `Trainer`, which is a class which combines the data and model together (just like Learner in fastai):

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_nm, num_labels=2)

trainer = Trainer(
    model, args, 
    train_dataset=dds['train'], 
    eval_dataset=dds['test'],
    tokenizer=tokz, 
    compute_metrics=my_metric
)

Now it's time to train the model!!

In [ ]:
trainer.train();

We can now use this model to generate some predictions on the test set

In [ ]:
preds = trainer.predict(tst_tok_ds)
preds

In [ ]:
preds.predictions
class_preds = np.argmax(preds.predictions, axis=-1)
class_preds

Now let's check to make sure our submission is in the same format as `submission.csv`

In [ ]:
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
sub

In [ ]:
sub['target'] = class_preds
sub

Now let's save our predictions ready for submission

In [ ]:
name = os.path.basename(model_nm)
sub.to_csv(f'submission_{name}_ep{epochs}.csv', index=False)  
sub.to_csv(f'submission.csv', index=False)